# Google ADK Practice
## Section 1: Environment Setup

Before we can build our agents, we need to install the necessary packages and configure our environment. Our agents will be powered by an **Azure OpenAI** deployment, connected through ADK's LiteLLM model connector.

### 1.1: Install Google ADK

First, let's install the Agent Development Kit (with the `mcp` extra used in Section 3), **LiteLLM** (ADK's connector for non-Gemini models such as Azure OpenAI), and `python-dotenv`.

**Note:** Recent LiteLLM versions require the MCP SDK 2.x (`mcp>=2.2`) when agents use tools. CrewAI currently pins `mcp` 1.x, so run this notebook and the CrewAI notebook in separate environments/kernels.

In [1]:
%pip install -q "google-adk[mcp]>=2.12" "litellm>=1.84" "mcp>=2.2" python-dotenv

# Windows only: pywin32 (used by the MCP SDK) normally needs a kernel restart after installation.
# Re-reading the site-packages .pth files makes it importable right away.
import site, sys, sysconfig
if sys.platform == "win32":
    for _site_dir in (sysconfig.get_paths()["purelib"], site.getusersitepackages()):
        site.addsitedir(_site_dir)

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.


### 1.2: Configure Azure OpenAI Credentials

Store your Azure OpenAI credentials in a `.env` file (in this folder or any parent folder). If a value is missing, this cell securely prompts for it.
```
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/openai/v1/
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [1]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


### 1.3: Import ADK Components

Now let's import the core building blocks we'll need:
- **LlmAgent** - The agent that orchestrates everything
- **LiteLlm** - Model connector that lets ADK use non-Gemini models such as Azure OpenAI (the brain of the agent)
- **FunctionTool** - Wraps Python functions as tools
- **InMemoryRunner** - Executes the agent
- **MCP Components** - For connecting to external services

In [2]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.adk.tools.function_tool import FunctionTool

# MCP imports for Section 3
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

print("✅ All ADK components imported successfully!")

✅ All ADK components imported successfully!


### 1.4: Configure the Model and Retry Options

ADK connects to Azure OpenAI through `LiteLlm`. Azure OpenAI's v1 API is OpenAI-compatible, so we use the `openai/<deployment>` model string with `api_base` pointing at the Azure endpoint.

When working with LLMs, network issues or rate limits can occur. `num_retries` tells LiteLLM to automatically retry failed calls (for example, rate limits or transient server errors).

In [3]:
azure_model = LiteLlm(
    model=f"openai/{AZURE_OPENAI_DEPLOYMENT}",
    api_base=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    num_retries=5,
)

print("✅ Azure OpenAI model configured with retries!")

✅ Azure OpenAI model configured with retries!


---
## Section 2: Building Your First Agent with Custom Tools

Let's create a **Recipe Finder Agent** that can search for recipes based on ingredients you have at home.

### The Agent Creation Flow:
```
1. Define a Python function (your custom tool)
2. Wrap it with FunctionTool
3. Create an LlmAgent with the tool
4. Create a Runner to execute the agent
5. Test with queries!
```

### 2.1: Define Your Custom Tool Function

First, we create a Python function that will act as a tool. The function should:
- Have clear parameters with type hints
- Include a descriptive docstring (the agent reads this!)
- Return structured data (dict, list, or string)

**Important:** The agent uses your docstring to understand WHEN and HOW to call this function!

In [4]:
def find_recipe(ingredients: str, cuisine: str = "any") -> dict:
    """Finds a recipe based on available ingredients and optional cuisine preference.

    Args:
        ingredients: Comma-separated list of ingredients available (e.g., "chicken, tomatoes, garlic")
        cuisine: Preferred cuisine type (e.g., "Italian", "Mexican", "Asian") or "any"

    Returns:
        Dictionary with recipe details including name, ingredients needed, and instructions
    """

    # Simulated recipe database (in production, this would be a real API call)
    recipes = {
        "chicken": {
            "name": "Garlic Chicken Pasta",
            "cuisine": "Italian",
            "ingredients": ["chicken", "pasta", "garlic", "olive oil", "parmesan"],
            "instructions": "1. Cook pasta. 2. Sauté garlic and chicken. 3. Combine with pasta and cheese.",
            "prep_time": "25 minutes",
            "servings": 4
        },
        "tomato": {
            "name": "Fresh Tomato Salsa",
            "cuisine": "Mexican",
            "ingredients": ["tomatoes", "onion", "cilantro", "lime", "jalapeño"],
            "instructions": "1. Dice tomatoes and onion. 2. Mix with chopped cilantro. 3. Add lime juice.",
            "prep_time": "10 minutes",
            "servings": 6
        },
        "egg": {
            "name": "Classic Omelet",
            "cuisine": "French",
            "ingredients": ["eggs", "butter", "cheese", "herbs"],
            "instructions": "1. Beat eggs. 2. Melt butter in pan. 3. Pour eggs and fold with fillings.",
            "prep_time": "8 minutes",
            "servings": 2
        }
    }

    # Simple ingredient matching logic
    ingredient_list = [ing.strip().lower() for ing in ingredients.split(",")]

    for key_ingredient in ingredient_list:
        for recipe_key, recipe in recipes.items():
            if recipe_key in key_ingredient:
                # Check cuisine preference
                if cuisine.lower() != "any" and cuisine.lower() not in recipe["cuisine"].lower():
                    continue
                return {
                    "status": "found",
                    "recipe": recipe
                }

    return {
        "status": "not_found",
        "message": f"No recipes found for ingredients: {ingredients}"
    }

print("✅ Recipe finder function created!")

✅ Recipe finder function created!


### 2.2: Create the Agent with Your Tool

Now we'll create an agent and give it access to our `find_recipe` tool.

**Key Components:**
- `model`: The LLM that powers the agent (our Azure OpenAI deployment via `LiteLlm`)
- `name`: Identifier for your agent
- `instruction`: The agent's system prompt - tells it HOW to behave
- `tools`: List of tools the agent can use (wrapped with `FunctionTool`)

In [5]:
recipe_agent = LlmAgent(
    model=azure_model,
    name="recipe_finder_agent",
    instruction="""You are a helpful cooking assistant that helps users find recipes.

    When a user asks for recipe suggestions:
    1. Use the find_recipe tool with the ingredients they mention
    2. Present the recipe in a friendly, easy-to-read format
    3. Highlight the prep time and number of servings
    4. If no recipe is found, suggest alternative ingredients or cuisines

    Keep your responses warm and encouraging!""",
    tools=[FunctionTool(func=find_recipe)]
)

print("✅ Recipe Finder Agent created!")

✅ Recipe Finder Agent created!


### 2.3: Create a Runner to Execute the Agent

The **Runner** is responsible for executing your agent. Think of it as the engine that runs the agent.

We're using `InMemoryRunner` - the simplest runner type, perfect for testing and development.

In [6]:
recipe_runner = InMemoryRunner(agent=recipe_agent)

print("✅ Runner initialized!")

✅ Runner initialized!


### 2.4: Test Your Agent!

Now let's test our agent by asking it to find recipes.

**Notice:** We use `await` because ADK operations are asynchronous.

In [7]:
# Test 1: Simple ingredient search
response = await recipe_runner.run_debug("I have chicken and garlic. What can I make?")

# Extract and print the response
for event in response:
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(part.text)

recipe_finder_agent > You can make **Garlic Chicken Pasta** — a quick, cozy Italian-style meal.

**Prep time:** 25 minutes  
**Servings:** 4

### Ingredients
- Chicken
- Garlic
- Pasta
- Olive oil
- Parmesan

### Instructions
1. Cook the pasta.
2. Sauté the garlic and chicken in olive oil.
3. Combine everything with the pasta and top with parmesan.

If you want, I can also suggest a **one-pan version** or give you a **recipe with only the ingredients you already have**.
You can make **Garlic Chicken Pasta** — a quick, cozy Italian-style meal.

**Prep time:** 25 minutes  
**Servings:** 4

### Ingredients
- Chicken
- Garlic
- Pasta
- Olive oil
- Parmesan

### Instructions
1. Cook the pasta.
2. Sauté the garlic and chicken in olive oil.
3. Combine everything with the pasta and top with parmesan.

If you want, I can also suggest a **one-pan version** or give you a **recipe with only the ingredients you already have**.


In [15]:
# Test 2: Cuisine preference
response = await recipe_runner.run_debug("I want to make something Mexican with tomatoes")

for event in response:
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(part.text)

recipe_finder_agent > Here’s a great Mexican option with tomatoes:

## Fresh Tomato Salsa

**Prep time:** 10 minutes  
**Servings:** 6

### Ingredients
- Tomatoes
- Onion
- Cilantro
- Lime
- Jalapeño

### Instructions
1. Dice the tomatoes and onion.
2. Mix with chopped cilantro.
3. Add lime juice.

It’s perfect with tortilla chips, tacos, or grilled meats. If you want, I can also suggest a **heartier Mexican tomato dish** like soup, rice, or enchiladas.
Here’s a great Mexican option with tomatoes:

## Fresh Tomato Salsa

**Prep time:** 10 minutes  
**Servings:** 6

### Ingredients
- Tomatoes
- Onion
- Cilantro
- Lime
- Jalapeño

### Instructions
1. Dice the tomatoes and onion.
2. Mix with chopped cilantro.
3. Add lime juice.

It’s perfect with tortilla chips, tacos, or grilled meats. If you want, I can also suggest a **heartier Mexican tomato dish** like soup, rice, or enchiladas.


---
## 🌐 Section 3: Model Context Protocol (MCP) Integration

Now let's level up! Instead of writing all tools ourselves, we can connect to **MCP Servers** - pre-built services that provide specialized tools.

### What is MCP?

**Model Context Protocol** is an open standard that connects agents to external services:

```
Your Agent <--MCP--> External Servers (GitHub, Slack, etc.)
```

**Benefits:**
- ✅ No need to write API integration code
- ✅ Community-maintained tools
- ✅ Standardized interface
- ✅ Easy to swap services

### 3.1: Adding Image Generation with MCP

Let's enhance our Recipe Assistant with the ability to generate food images using the **Everything MCP Server**.

**MCP Setup Flow:**
```
1. Choose an MCP server
2. Configure connection parameters
3. Create McpToolset
4. Add to agent's tools
```

### 3.2: Configure the MCP Toolset

We'll connect to the **Everything MCP Server** which provides a `get-tiny-image` tool.

**Connection Details:**
- Uses `npx` to run the MCP server (requires **Node.js 24+**; the MCP cells are skipped if `npx` is not found)
- Server package: `@modelcontextprotocol/server-everything`
- We filter to only use the `get-tiny-image` tool

In [20]:
import shutil
import sys

NPX_AVAILABLE = shutil.which("npx") is not None

mcp_image_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=[
                "-y",
                "@modelcontextprotocol/server-everything",
            ],
        ),
        timeout=120,  # the first run downloads the server package via npx
    ),
    tool_filter=["get-tiny-image"],
    # Jupyter's sys.stderr has no OS file handle on Windows, so give the server process the real one
    errlog=sys.__stderr__,
)

if NPX_AVAILABLE:
    print("✅ MCP Toolset configured!")
else:
    print("⚠️ MCP Toolset configured, but npx was not found. Install Node.js 24+ to run the MCP cells.")

✅ MCP Toolset configured!


### 3.3: Create Enhanced Agent with Both Tools

Now let's create an enhanced agent that has BOTH our custom `find_recipe` tool AND the MCP image generation tool.

In [21]:
enhanced_recipe_agent = LlmAgent(
    model=azure_model,
    name="enhanced_recipe_agent",
    instruction="""You are an advanced cooking assistant with image generation capabilities.

    When a user asks for recipes:
    1. Use find_recipe to search for recipes
    2. Present the recipe clearly
    3. If the user wants visual inspiration, use get-tiny-image to generate a sample image

    Always be helpful and creative!""",
    tools=[
        FunctionTool(func=find_recipe),
        mcp_image_toolset
    ]
)

enhanced_runner = InMemoryRunner(agent=enhanced_recipe_agent)

print("✅ Enhanced agent with MCP tool created!")

✅ Enhanced agent with MCP tool created!


### 3.4: Test the Enhanced Agent

Let's ask for a recipe with an image! ADK starts the MCP server (via `npx`) the first time the agent needs its tools.

In [22]:
if NPX_AVAILABLE:
    response = await enhanced_runner.run_debug(
        "I have eggs. Can you suggest a recipe and show me a sample image?"
    )

    # Display text responses
    for event in response:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)
else:
    print("Skipped: the MCP server needs Node.js 24+ (npx).")

enhanced_recipe_agent > Sure — here’s a simple recipe using eggs:

## Classic Omelet
**Cuisine:** French  
**Prep time:** 8 minutes  
**Servings:** 2

### Ingredients
- Eggs
- Butter
- Cheese
- Herbs

### Instructions
1. Beat the eggs.
2. Melt butter in a pan over medium heat.
3. Pour in the eggs.
4. Add cheese and herbs.
5. Fold the omelet and cook until set.

### Sample image
I’ve shown you a sample image above.

If you want, I can also suggest:
- a **2-ingredient egg recipe**
- a **breakfast recipe**
- or a recipe using **other ingredients you have**
Sure — here’s a simple recipe using eggs:

## Classic Omelet
**Cuisine:** French  
**Prep time:** 8 minutes  
**Servings:** 2

### Ingredients
- Eggs
- Butter
- Cheese
- Herbs

### Instructions
1. Beat the eggs.
2. Melt butter in a pan over medium heat.
3. Pour in the eggs.
4. Add cheese and herbs.
5. Fold the omelet and cook until set.

### Sample image
I’ve shown you a sample image above.

If you want, I can also suggest:
- a **2-ingr

### 3.5: Display Generated Images

The MCP server returns base64-encoded images. Let's decode and display them!

In [23]:
from IPython.display import display, Image as IPImage
import base64

if NPX_AVAILABLE:
    # Extract and display any images from the response
    for event in response:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "function_response") and part.function_response:
                    for item in part.function_response.response.get("content", []):
                        if item.get("type") == "image":
                            print("\n🖼️ Generated Image:")
                            display(IPImage(data=base64.b64decode(item["data"])))
else:
    print("Skipped: the MCP server needs Node.js 24+ (npx).")


🖼️ Generated Image:


---
## Section 4: Observability - Debugging Your Agents

When building agents, you need to understand what's happening under the hood:
- Which tools are being called?
- What are the tool inputs and outputs?
- How is the agent reasoning?

ADK provides powerful observability features to help you debug and monitor your agents.

### 4.1: Verbose Mode

The simplest way to see what's happening is to use `verbose=True` in `run_debug()`.

This prints:
- 📝 User messages
- 🤖 Agent reasoning
- 🛠️ Tool calls with parameters
- 📊 Tool responses

In [24]:
print("=" * 60)
print("🔍 VERBOSE MODE DEMONSTRATION")
print("=" * 60)

response = await recipe_runner.run_debug(
    "Find me an Italian recipe with chicken",
    verbose=True  # This enables detailed logging
)

🔍 VERBOSE MODE DEMONSTRATION
recipe_finder_agent > [Calling tool: find_recipe({'ingredients': 'chicken', 'cuisine': 'Italian'})]
recipe_finder_agent > [Tool result: {'status': 'found', 'recipe': {'name': 'Garlic Chicken Pasta', 'cuisine': 'Italian', 'ingredients': ...]
recipe_finder_agent > Here’s a tasty Italian recipe with chicken:

## Garlic Chicken Pasta

**Prep time:** 25 minutes  
**Servings:** 4

### Ingredients
- Chicken
- Pasta
- Garlic
- Olive oil
- Parmesan

### Instructions
1. Cook the pasta.
2. Sauté garlic and chicken.
3. Combine with pasta and cheese.

If you'd like, I can also find a **different Italian chicken recipe** or suggest one that uses **fewer ingredients**.


### 4.2: Event Streaming and Inspection

For more control, you can inspect individual events. Each event contains:
- `event.content` - The message content
- `event.content.parts` - Parts of the message (text, function calls, etc.)
- Metadata like timestamps and invocation IDs

In [25]:
def analyze_events(events):
    """Helper function to analyze and categorize events."""

    text_responses = []
    tool_calls = []
    tool_responses = []

    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                # Text responses from agent
                if part.text:
                    text_responses.append(part.text)

                # Tool calls made by agent
                if hasattr(part, 'function_call') and part.function_call:
                    tool_calls.append({
                        'name': part.function_call.name,
                        'args': part.function_call.args
                    })

                # Tool responses
                if hasattr(part, 'function_response') and part.function_response:
                    tool_responses.append({
                        'name': part.function_response.name,
                        'response': part.function_response.response
                    })

    return text_responses, tool_calls, tool_responses

print("✅ Event analyzer function created!")

✅ Event analyzer function created!


### 4.3: Test Event Analysis

Let's run a query and analyze all the events that occur.

In [26]:
response = await recipe_runner.run_debug("What can I cook with tomatoes?")

text_responses, tool_calls, tool_responses = analyze_events(response)

print("\n" + "=" * 60)
print("📊 EVENT ANALYSIS")
print("=" * 60)

print(f"\n🛠️ TOOL CALLS ({len(tool_calls)}):")
for i, call in enumerate(tool_calls, 1):
    print(f"  {i}. {call['name']}")
    print(f"     Args: {call['args']}")

print(f"\n📊 TOOL RESPONSES ({len(tool_responses)}):")
for i, resp in enumerate(tool_responses, 1):
    print(f"  {i}. {resp['name']}")
    print(f"     Response: {resp['response']}")

print(f"\n💬 AGENT TEXT RESPONSES ({len(text_responses)}):")
for i, text in enumerate(text_responses, 1):
    print(f"  {i}. {text[:100]}...")  # First 100 chars

print("\n" + "=" * 60)

recipe_finder_agent > You can make **Fresh Tomato Salsa**.

**Cuisine:** Mexican  
**Prep time:** 10 minutes  
**Servings:** 6

### Ingredients
- Tomatoes
- Onion
- Cilantro
- Lime
- Jalapeño

### Instructions
1. Dice the tomatoes and onion.
2. Mix with chopped cilantro.
3. Add lime juice.

Great with chips, tacos, or as a topping for grilled chicken. If you want, I can also suggest **more tomato recipes** like pasta sauce, soup, or salad.

📊 EVENT ANALYSIS

🛠️ TOOL CALLS (1):
  1. find_recipe
     Args: {'ingredients': 'tomatoes', 'cuisine': 'any'}

📊 TOOL RESPONSES (1):
  1. find_recipe
     Response: {'status': 'found', 'recipe': {'name': 'Fresh Tomato Salsa', 'cuisine': 'Mexican', 'ingredients': ['tomatoes', 'onion', 'cilantro', 'lime', 'jalapeño'], 'instructions': '1. Dice tomatoes and onion. 2. Mix with chopped cilantro. 3. Add lime juice.', 'prep_time': '10 minutes', 'servings': 6}}

💬 AGENT TEXT RESPONSES (1):
  1. You can make **Fresh Tomato Salsa**.

**Cuisine:** Mexican  
**

### 4.4: Custom Logging for Production

In production, you might want to log events to a file or monitoring service. Here's a pattern you can use:

In [27]:
import json
from datetime import datetime

def log_agent_execution(query: str, events):
    """Log agent execution for monitoring and debugging."""

    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "events": []
    }

    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    log_entry["events"].append({
                        "type": "text",
                        "content": part.text
                    })
                if hasattr(part, 'function_call') and part.function_call:
                    log_entry["events"].append({
                        "type": "tool_call",
                        "tool": part.function_call.name,
                        "args": dict(part.function_call.args)
                    })

    # In production, you'd write to a logging service
    print("\n📝 LOG ENTRY:")
    print(json.dumps(log_entry, indent=2))

    return log_entry

# Test it
response = await recipe_runner.run_debug("Quick egg recipe")
log = log_agent_execution("Quick egg recipe", response)

recipe_finder_agent > A great quick option is **Classic Omelet**.

**Prep time:** 8 minutes  
**Servings:** 2

### Ingredients
- Eggs
- Butter
- Cheese
- Herbs

### Instructions
1. Beat the eggs.
2. Melt butter in a pan.
3. Pour in the eggs and fold with fillings.

If you want, I can also suggest an **even faster egg recipe** like scrambled eggs or a mug omelet.

📝 LOG ENTRY:
{
  "timestamp": "2026-11-23T13:42:51.814705",
  "query": "Quick egg recipe",
  "events": [
    {
      "type": "tool_call",
      "tool": "find_recipe",
      "args": {
        "ingredients": "egg",
        "cuisine": "any"
      }
    },
    {
      "type": "text",
      "content": "A great quick option is **Classic Omelet**.\n\n**Prep time:** 8 minutes  \n**Servings:** 2\n\n### Ingredients\n- Eggs\n- Butter\n- Cheese\n- Herbs\n\n### Instructions\n1. Beat the eggs.\n2. Melt butter in a pan.\n3. Pour in the eggs and fold with fillings.\n\nIf you want, I can also suggest an **even faster egg recipe** like scramble

---
## 🎯 Section 5: Final Challenge - Build Your Own Agent!

Now it's your turn to apply everything you've learned!

## 🍕 The Challenge: Nutrition Calculator Agent

Build an agent that helps users track nutritional information for their meals.

### Requirements:

1. **Create a custom tool:** `calculate_nutrition(food_item: str, serving_size: float) -> dict`
   - Should return calories, protein, carbs, and fat for common foods
   - Create a simple database with at least 5 food items
   - Example foods: rice, chicken breast, broccoli, apple, salmon

2. **Create the agent:**
   - Use the Azure OpenAI model (`azure_model`)
   - Give it clear instructions on how to help users
   - Add the nutrition calculator tool

3. **Test your agent:**
   - Ask it to calculate nutrition for a single food
   - Ask it to calculate total nutrition for a meal with multiple items
   - Use verbose mode to see the tool calls

4. **BONUS:** Add the MCP image tool so users can visualize their meals!

### Starter Code Structure:

```python
# Step 1: Define your tool
def calculate_nutrition(food_item: str, serving_size: float = 1.0) -> dict:
    # Your code here
    pass

# Step 2: Create the agent
nutrition_agent = LlmAgent(
    # Your configuration here
)

# Step 3: Create runner
nutrition_runner = InMemoryRunner(agent=nutrition_agent)

# Step 4: Test it
response = await nutrition_runner.run_debug("Calculate nutrition for 200g chicken breast", verbose=True)
```

### Tips:
- Look back at the recipe agent for patterns
- Use clear docstrings - the agent reads them!
- Test with verbose=True to debug
- Start simple, then enhance

**Good luck! 🚀**

In [30]:
# YOUR CODE HERE - Build the nutrition calculator tool
# Nutrition database (values per 100g)
nutrition_database = {
    "rice": {"calories": 130, "protein": 2.7, "carbs": 28.0, "fat": 0.3},
    "chicken breast": {"calories": 165, "protein": 31.0, "carbs": 0.0, "fat": 3.6},
    "broccoli": {"calories": 34, "protein": 2.8, "carbs": 6.6, "fat": 0.4},
    "apple": {"calories": 52, "protein": 0.3, "carbs": 13.8, "fat": 0.2},
    "salmon": {"calories": 208, "protein": 20.4, "carbs": 0.0, "fat": 13.4},
}


def calculate_nutrition(food_item: str, serving_size: float = 1.0) -> dict:
    """Calculate calories, protein, carbs, and fat for a common food.
    Nutrition values are stored per 100g; serving_size is in grams.
    """
    food_item = food_item.lower().strip()

    if food_item not in nutrition_database:
        return {
            "error": f"{food_item} is not in the database.",
            "available_foods": list(nutrition_database.keys())
        }

    if serving_size <= 0:
        return {"error": "Serving size must be greater than 0 grams."}

    food = nutrition_database[food_item]
    multiplier = serving_size / 100

    return {
        "food_item": food_item,
        "serving_size_g": serving_size,
        "calories": round(food["calories"] * multiplier, 2),
        "protein": round(food["protein"] * multiplier, 2),
        "carbs": round(food["carbs"] * multiplier, 2),
        "fat": round(food["fat"] * multiplier, 2)
    }
calculate_nutrition("chicken breast", 200)

{'food_item': 'chicken breast',
 'serving_size_g': 200,
 'calories': 330.0,
 'protein': 62.0,
 'carbs': 0.0,
 'fat': 7.2}

In [33]:
# YOUR CODE HERE - Create the nutrition agent
# Create the nutrition agent
nutrition_agent = LlmAgent(
    name="nutrition_agent",
    model=azure_model,
    description="An agent that calculates nutritional values for foods and meals.",
    instruction="""
    You are a nutrition calculator assistant.

    Use the calculate_nutrition tool to calculate nutritional values.
    Do not estimate values yourself.

    For a single food:
    - Report calories, protein, carbohydrates, and fat.
    - Use the serving size in grams provided by the user.

    For meals with multiple foods:
    - Calculate each food separately using the tool.
    - Add the results together to provide total nutrition values.
    - Show both individual food values and meal totals.

    Always clearly state that nutritional values are approximate.
    """,
    tools=[calculate_nutrition]
)

In [35]:
nutrition_runner = InMemoryRunner(agent=nutrition_agent)

In [ ]:
# YOUR CODE HERE - Test your agent
# Test with a single food
response = await nutrition_runner.run_debug(
    "Calculate nutrition for 200g chicken breast",
    verbose=True
)
# Test with multiple foods
response = await nutrition_runner.run_debug(
    "Calculate the total nutrition for a meal with 150g rice, 200g chicken breast, and 100g broccoli. Show individual items and the total.",
    verbose=True
)

# Test verbose nutrition analysis
response = await nutrition_runner.run_debug(
    "Calculate the nutrition for 200g salmon and 150g broccoli. Break down each ingredient, calculate the combined totals, and explain the nutritional contribution of each food.",
    verbose=True
)

nutrition_agent > [Calling tool: calculate_nutrition({'food_item': 'chicken breast', 'serving_size': 20...)]
nutrition_agent > [Tool result: {'food_item': 'chicken breast', 'serving_size_g': 200, 'calories': 330.0, 'protein': 62.0, 'carbs': ...]
nutrition_agent > Approximate nutrition for **200g chicken breast**:

- **Calories:** 330 kcal
- **Protein:** 62 g
- **Carbohydrates:** 0 g
- **Fat:** 7.2 g
nutrition_agent > [Calling tool: calculate_nutrition({'food_item': 'rice', 'serving_size': 150})]
nutrition_agent > [Calling tool: calculate_nutrition({'food_item': 'chicken breast', 'serving_size': 20...)]
nutrition_agent > [Calling tool: calculate_nutrition({'food_item': 'broccoli', 'serving_size': 100})]
nutrition_agent > [Tool result: {'food_item': 'rice', 'serving_size_g': 150, 'calories': 195.0, 'protein': 4.05, 'carbs': 42.0, 'fat...]
nutrition_agent > [Tool result: {'food_item': 'chicken breast', 'serving_size_g': 200, 'calories': 330.0, 'protein': 62.0, 'carbs': ...]
nutrition_agen

---
## 🎓 What You've Learned

Congratulations! You've mastered the fundamentals of Google ADK:

✅ **Creating Agents** - Built agents with custom tools using `LlmAgent`

✅ **Custom Tools** - Wrapped Python functions with `FunctionTool`

✅ **Bring Your Own Model** - Powered ADK agents with Azure OpenAI through `LiteLlm`

✅ **MCP Integration** - Connected to external services with `McpToolset`

✅ **Observability** - Debugged with verbose mode and event inspection

✅ **Production Patterns** - Learned error handling, retry logic, and logging

### 📚 Resources:

- [Google ADK Documentation](https://adk.dev/)
- [ADK LiteLLM Model Connector](https://adk.dev/agents/models/litellm/)
- [Azure OpenAI v1 API](https://learn.microsoft.com/azure/ai-foundry/openai/api-version-lifecycle)
- [MCP Protocol Specification](https://modelcontextprotocol.io/)

**Happy Building! 🎉**